### Model Context Protocol

In [1]:
from fastmcp import FastMCP
import json

In [2]:
mcp=FastMCP(
    "First Server"
)

In [3]:
# Tool
@mcp.tool
def add(a:int,b:int) -> int:
    """This recieves two numbers and return their sum"""
    return a+b

### Connect with FastMCP client

In [4]:
from fastmcp import Client

In [5]:
async with Client(mcp) as client:
    print("Protocol Version",client.protocol_version)
    tools=await client.list_tools()
    print("Tools",tools)

    print("Tool Calling")
    result=await client.call_tool("add",{'a':5,'b':3})
    print(result.data)

Protocol Version 2026-07-28
Tools [Tool(name='add', title='Add', description='This recieves two numbers and return their sum', input_schema={'type': 'object', 'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b']}, execution=None, output_schema={'properties': {'result': {'type': 'integer'}}, 'required': ['result'], 'type': 'object', 'x-fastmcp-wrap-result': True}, icons=None, annotations=None, meta={'fastmcp': {'tags': []}})]
Tool Calling
8


### MCP Tools -> Context, Tasks , Auth

In [6]:
async def show(server:FastMCP,tool_name:str):
    async with Client(server) as client:
        tool=None
        for t in await client.list_tools():
            if t.name==tool_name:
                tool=t
                break
        print(f"Tools:{tool.name}:{tool.description}")
        print(json.dumps(tool.input_schema,indent=2))

In [7]:
await show(mcp,'add')

Tools:add:This recieves two numbers and return their sum
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "a": {
      "type": "integer"
    },
    "b": {
      "type": "integer"
    }
  },
  "required": [
    "a",
    "b"
  ]
}


In [8]:
@mcp.tool
def multiply(a:int, b:int=1) -> int:
    """Takes two arguments and return their multiplication"""
    return a*b

In [9]:
await show(mcp,'multiply')

Tools:multiply:Takes two arguments and return their multiplication
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "a": {
      "type": "integer"
    },
    "b": {
      "default": 1,
      "type": "integer"
    }
  },
  "required": [
    "a"
  ]
}


In [10]:
from typing import Literal

In [11]:
@mcp.tool
def export(table:str,format:Literal['csv','json','excel']='csv'):
    """Return the table in the required format"""
    return 'Export Done'

In [12]:
await show(mcp,'export')

Tools:export:Return the table in the required format
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "table": {
      "type": "string"
    },
    "format": {
      "default": "csv",
      "enum": [
        "csv",
        "json",
        "excel"
      ],
      "type": "string"
    }
  },
  "required": [
    "table"
  ]
}


### Pydantic model as a Argument

In [13]:
from pydantic import BaseModel , Field

class Shipment(BaseModel):
    order_id:str=Field(description="This is the order id placed by the user")
    quantity:int=Field(description="This is the quantity")
    express:bool=False

@mcp.tool
def book_shipment(shipment:Shipment):
    """Book a shipment for a order"""
    return {"Booked":shipment.order_id,"qty":shipment.quantity}

In [14]:
await show(mcp,'book_shipment')

Tools:book_shipment:Book a shipment for a order
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "shipment": {
      "properties": {
        "order_id": {
          "description": "This is the order id placed by the user",
          "type": "string"
        },
        "quantity": {
          "description": "This is the quantity",
          "type": "integer"
        },
        "express": {
          "default": false,
          "type": "boolean"
        }
      },
      "required": [
        "order_id",
        "quantity"
      ],
      "type": "object"
    }
  },
  "required": [
    "shipment"
  ]
}


### Exception/Error Handling

In [15]:
from fastmcp.exceptions import ToolError

In [16]:
@mcp.tool
def new_charges(amount:float):
    """Charges on card"""
    if amount<100:
        raise ToolError("Amount should be greater than 100")
    return "Charges Done"

In [17]:
async with Client(mcp) as client:
    result=await client.call_tool('new_charges',{'amount':10},raise_on_error=False) 
    print(result)

[11/08/26 19:26:28] Error calling tool 'new_charges'

CallToolResult(content=[TextContent(type='text', text='Amount should be greater than 100', annotations=None, meta=None)], structured_content=None, meta={'io.modelcontextprotocol/serverInfo': {'name': 'First Server', 'version': '4.1.6'}}, data=None, is_error=True)


### Context -> Talking back to the client while the server is running 

In [22]:
import asyncio

from fastmcp import FastMCP,Context

ctxserver=FastMCP("Context")

@ctxserver.tool
async def send_mails(emails:list[str],context:Context) -> str:
    """Send Emails to many students at a time"""

    totalStudents=len(emails)

    # One email at a time, 2 sec to complete 
    for i in range(totalStudents):
        email=emails[i]
        await context.report_progress(progress=i+1,total=totalStudents,message=f"Sending email to : {email}")

        await asyncio.sleep(2)

    return "Email sent to all the students"

In [23]:
async def watch_email_process(progress,total,message):
    print(message)

async with Client(ctxserver) as client:
    res=await client.call_tool('send_mails',{'emails':['pratik','nikita','ajay']},progress_handler=watch_email_process)

    print(res)

Sending email to : pratik
Sending email to : nikita
Sending email to : ajay
CallToolResult(content=[TextContent(type='text', text='Email sent to all the students', annotations=None, meta=None)], structured_content={'result': 'Email sent to all the students'}, meta={'fastmcp': {'wrap_result': True}, 'io.modelcontextprotocol/serverInfo': {'name': 'Context', 'version': '4.1.6'}}, data='Email sent to all the students', is_error=False)


### Long Running Task

In [6]:
from fastmcp import FastMCP, Client
import fastmcp_tasks
from fastmcp.dependencies import Progress
from fastmcp.utilities.tasks import TaskConfig
from fastmcp_tasks import TasksExtension,call_tool_task
import asyncio

In [7]:
mcp=FastMCP('Task Server')

mcp.add_extension(TasksExtension())

@mcp.tool(task=TaskConfig(mode="required"))
async def slow_computation(email_count:int,progress:Progress=Progress())->str:
    """A long running task / operation"""
    await progress.set_total(email_count)
    for i in range(email_count):
        ### Sending the email code
        await asyncio.sleep(1)
        await progress.set_message(f'Sending email {i} out of {email_count}')
        await progress.increment()
    return f'Completed in {email_count}seconds..'

In [11]:
async with Client(mcp) as client:
    task=await call_tool_task(client,'slow_computation',{'email_count':10})
    print(f'Task id :',task.task_id)

    while True:
        status=await task.status()

        print('Status:',status.status_message)

        if status.status=='completed':
            break
        await asyncio.sleep(1)

Task id : EHkYX00ppTpqEmwNJ1q2XFXzyRFnTAhnk0-Rlej_DBU
Status: None
Status: Sending email 0 out of 10
Status: Sending email 1 out of 10
Status: Sending email 2 out of 10
Status: Sending email 3 out of 10
Status: Sending email 4 out of 10
Status: Sending email 5 out of 10
Status: Sending email 6 out of 10
Status: Sending email 7 out of 10
Status: Sending email 8 out of 10
Status: None


### JWT Authentication

### Server Code in auth_mcp_server.py

In [23]:
TOKEN_GENERATOR_URL='http://127.0.0.1:8001'
MCP_SERVER_URL='http://127.0.0.1:8000/mcp'

In [40]:
from fastmcp import Client
import httpx
import requests

async def getToken():
    async with httpx.AsyncClient() as http:
        res=await http.post(
            f'{TOKEN_GENERATOR_URL}/token',
            json={'user_id':'rahul','email':'rahulty.com','password':"1245"}
        )

        print(res.status_code)
        if res.status_code!=200:
            return 'No token found'

        data=res.json()

        token=data['access_token']
        return token

In [41]:
token=await getToken()

200


In [42]:
# token

In [43]:
async with Client(MCP_SERVER_URL,auth=token) as client:
    tools=await client.list_tools()
    result=await client.call_tool('whoami',{})
    print(result.data)

{'client_id': 'rahul', 'email': ''}
